In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

# ── Styling ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#F4F4F0',
    'axes.grid':        True,
    'grid.color':       'white',
    'grid.linewidth':   1.2,
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})
BLUE   = '#2563EB'
PURPLE = '#7C3AED'
CORAL  = '#E85D2F'
TEAL   = '#0F766E'
AMBER  = '#D97706'
GRAY   = '#6B7280'

df = pd.read_csv('raw.csv')
df['post_datetime'] = pd.to_datetime(df['post_datetime'])
df['hour']       = df['post_datetime'].dt.hour
df['day_of_week']= df['post_datetime'].dt.day_name()
df['log_views']  = np.log1p(df['views'])

print("Dataset loaded:", df.shape)
print(df.dtypes)

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — DESCRIPTIVE ANALYTICS
# ══════════════════════════════════════════════════════════════════════════════

# ── Fig 1 : Distribution of Views ────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 1 — Distribution of Views', fontsize=15, fontweight='bold', y=1.01)

ax = axes[0]
ax.hist(df['views'], bins=50, color=BLUE, edgecolor='white', linewidth=0.5)
ax.axvline(df['views'].mean(),   color=CORAL,  lw=2, linestyle='--', label=f"Mean   {df['views'].mean()/1e6:.2f}M")
ax.axvline(df['views'].median(), color=AMBER,  lw=2, linestyle='-',  label=f"Median {df['views'].median()/1e6:.2f}M")
ax.set_title('Raw views (right-skewed)')
ax.set_xlabel('Views'); ax.set_ylabel('Frequency')
ax.legend()
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))

ax2 = axes[1]
ax2.hist(df['log_views'], bins=50, color=TEAL, edgecolor='white', linewidth=0.5)
ax2.axvline(df['log_views'].mean(),   color=CORAL, lw=2, linestyle='--', label=f"Mean {df['log_views'].mean():.2f}")
ax2.axvline(df['log_views'].median(), color=AMBER, lw=2, linestyle='-',  label=f"Median {df['log_views'].median():.2f}")
ax2.set_title('Log-transformed views (approx. normal)')
ax2.set_xlabel('log(views)'); ax2.set_ylabel('Frequency')
ax2.legend()

# Annotation box
stats_text = (
    f"Mean:   {df['views'].mean():,.0f}\n"
    f"Median: {df['views'].median():,.0f}\n"
    f"Std:    {df['views'].std():,.0f}\n"
    f"Skew:   {df['views'].skew():.2f}\n"
    f"IQR:    {(df['views'].quantile(.75)-df['views'].quantile(.25)):,.0f}"
)
axes[0].text(0.97, 0.97, stats_text, transform=axes[0].transAxes,
             va='top', ha='right', fontsize=9, family='monospace',
             bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig('fig1_distribution.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 1 saved")

# ── Fig 2 : Segmented Summaries ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Phase 1 — Average Views by Platform & Content Type', fontsize=15, fontweight='bold')

plat = df.groupby('platform')['views'].mean().sort_values(ascending=False)
colors_p = [BLUE, PURPLE, CORAL, TEAL]
bars = axes[0].bar(plat.index, plat.values/1e6, color=colors_p, edgecolor='white', linewidth=0.5)
axes[0].set_title('By Platform')
axes[0].set_xlabel('Platform'); axes[0].set_ylabel('Avg Views (Millions)')
for bar, val in zip(bars, plat.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val/1e6:.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')

ct = df.groupby('content_type')['views'].mean().sort_values(ascending=False)
colors_c = [AMBER, BLUE, TEAL, CORAL, PURPLE][:len(ct)]
bars2 = axes[1].bar(ct.index, ct.values/1e6, color=colors_c, edgecolor='white', linewidth=0.5)
axes[1].set_title('By Content Type')
axes[1].set_xlabel('Content Type'); axes[1].set_ylabel('Avg Views (Millions)')
for bar, val in zip(bars2, ct.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
                 f'{val/1e6:.2f}M', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig2_segmented.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 2 saved")

# ── Fig 3 : Correlation Heatmap ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
fig.suptitle('Phase 1 — Pearson Correlation Matrix', fontsize=15, fontweight='bold')
num_cols = ['views','likes','comments','shares','engagement_rate','sentiment_score']
corr = df[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlBu_r',
            center=0, square=True, linewidths=0.5, ax=ax,
            annot_kws={'size':10,'weight':'bold'})
ax.set_title('Lower triangle — Pearson r values', fontsize=10)
plt.tight_layout()
plt.savefig('fig3_heatmap.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 3 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — INFERENTIAL ANALYTICS
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("PHASE 2 — INFERENTIAL ANALYTICS")
print("="*60)

# ── T-Test : Video vs Non-Video ───────────────────────────────────────────────
video     = df[df['content_type']=='video']['views']
non_video = df[df['content_type']!='video']['views']
t_stat, p_val = stats.ttest_ind(video, non_video)
print(f"\n[T-TEST] Video vs Non-Video views")
print(f"  H₀: Mean views are equal for video and non-video content")
print(f"  H₁: Mean views differ between video and non-video content")
print(f"  Video mean:     {video.mean():,.0f}")
print(f"  Non-video mean: {non_video.mean():,.0f}")
print(f"  t-statistic: {t_stat:.4f}  |  p-value: {p_val:.4f}")
print(f"  Result: {'REJECT H₀ (significant)' if p_val<0.05 else 'FAIL to reject H₀ (not significant)'} at α=0.05")

# ── ANOVA : Views across Platforms ───────────────────────────────────────────
groups = [grp['views'].values for _, grp in df.groupby('platform')]
f_stat, p_anova = stats.f_oneway(*groups)
print(f"\n[ANOVA] Views across Platforms")
print(f"  H₀: Mean views are equal across all platforms")
print(f"  H₁: At least one platform has a significantly different mean")
print(f"  F-statistic: {f_stat:.4f}  |  p-value: {p_anova:.4f}")
print(f"  Result: {'REJECT H₀ (significant)' if p_anova<0.05 else 'FAIL to reject H₀'} at α=0.05")
plat_means = df.groupby('platform')['views'].mean().sort_values(ascending=False)
print(f"  Platform means:\n{plat_means.to_string()}")

# ── 95% CI : Viral vs Non-Viral ───────────────────────────────────────────────
viral     = df[df['is_viral']==1]['views']
non_viral = df[df['is_viral']==0]['views']

def ci95(series):
    n, m, s = len(series), series.mean(), series.std()
    se = s / np.sqrt(n)
    margin = stats.t.ppf(0.975, df=n-1) * se
    return m, m-margin, m+margin

v_mean, v_lo, v_hi   = ci95(viral)
nv_mean, nv_lo, nv_hi = ci95(non_viral)
print(f"\n[95% CONFIDENCE INTERVALS]")
print(f"  Viral     (n={len(viral):4d}): mean={v_mean:,.0f}  CI=[{v_lo:,.0f}, {v_hi:,.0f}]")
print(f"  Non-viral (n={len(non_viral):4d}): mean={nv_mean:,.0f}  CI=[{nv_lo:,.0f}, {nv_hi:,.0f}]")

# ── Fig 4 : Inferential Charts ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Phase 2 — Inferential Analytics', fontsize=15, fontweight='bold')

# T-test bar
ax = axes[0]
means = [video.mean()/1e6, non_video.mean()/1e6]
sems  = [video.sem()/1e6,  non_video.sem()/1e6]
bars  = ax.bar(['Video','Non-Video'], means, yerr=sems, color=[CORAL, BLUE],
               edgecolor='white', capsize=6, linewidth=0.5)
ax.set_title(f'T-Test: Video vs Non-Video\np = {p_val:.4f}', fontsize=10)
ax.set_ylabel('Mean Views (Millions)')
for bar, val in zip(bars, means):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+sems[bars.index(bar)]+0.05,
            f'{val:.2f}M', ha='center', fontsize=9, fontweight='bold')
sig_label = '*** Significant' if p_val < 0.05 else 'Not significant'
ax.text(0.5, 0.95, sig_label, transform=ax.transAxes, ha='center', va='top',
        color=CORAL if p_val < 0.05 else GRAY, fontweight='bold', fontsize=10)

# ANOVA bar
ax2 = axes[1]
pm = df.groupby('platform')['views'].mean().sort_values(ascending=False)/1e6
ps = df.groupby('platform')['views'].sem().reindex(pm.index)/1e6
bars2 = ax2.bar(pm.index, pm.values, yerr=ps.values, color=[BLUE,PURPLE,CORAL,TEAL],
                edgecolor='white', capsize=6, linewidth=0.5)
ax2.set_title(f'ANOVA: Views by Platform\np = {p_anova:.4f}', fontsize=10)
ax2.set_ylabel('Mean Views (Millions)')
ax2.tick_params(axis='x', rotation=15)
sig_label2 = '*** Significant' if p_anova < 0.05 else 'Not significant'
ax2.text(0.5, 0.95, sig_label2, transform=ax2.transAxes, ha='center', va='top',
         color=CORAL if p_anova < 0.05 else GRAY, fontweight='bold', fontsize=10)

# CI plot
ax3 = axes[2]
labels   = ['Viral', 'Non-Viral']
means_ci = [v_mean/1e6,  nv_mean/1e6]
lo_ci    = [v_lo/1e6,    nv_lo/1e6]
hi_ci    = [v_hi/1e6,    nv_hi/1e6]
ax3.barh(labels, means_ci, color=[CORAL, BLUE], edgecolor='white', linewidth=0.5, height=0.4)
for i,(lo,hi,m) in enumerate(zip(lo_ci,hi_ci,means_ci)):
    ax3.errorbar(m, i, xerr=[[m-lo],[hi-m]], fmt='none', color='black', capsize=6, lw=2)
ax3.set_title('95% CI: Viral vs Non-Viral', fontsize=10)
ax3.set_xlabel('Mean Views (Millions)')

plt.tight_layout()
plt.savefig('fig4_inferential.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 4 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  PHASE 3 — PREDICTIVE ANALYTICS (Multiple Linear Regression)
# ══════════════════════════════════════════════════════════════════════════════

print("\n" + "="*60)
print("PHASE 3 — PREDICTIVE ANALYTICS")
print("="*60)

# Prepare features
df_model = df.copy()
df_model = pd.get_dummies(df_model, columns=['platform','content_type','region'], drop_first=True)
bool_cols = df_model.select_dtypes(include='bool').columns
df_model[bool_cols] = df_model[bool_cols].astype(int)

feature_cols = ['likes','shares','comments','sentiment_score','is_viral'] + \
               [c for c in df_model.columns if c.startswith(('platform_','content_type_','region_'))]

X = df_model[feature_cols]
y = df_model['log_views']   # using log-transformed views
X_const = sm.add_constant(X)

model = sm.OLS(y, X_const).fit()
print(model.summary())

# Print equation
print("\n── REGRESSION EQUATION ──────────────────────────────────")
print(f"  Predicted log(views) = {model.params['const']:.4f}")
for feat in ['likes','shares','comments','sentiment_score','is_viral']:
    coef = model.params[feat]
    pv   = model.pvalues[feat]
    sig  = '***' if pv<0.001 else ('**' if pv<0.01 else ('*' if pv<0.05 else ''))
    print(f"    + {coef:.6f} × {feat:<20s}  {sig}")
print(f"\n  R² = {model.rsquared:.4f}  ({model.rsquared*100:.1f}% of variance explained)")
print(f"  Adj R² = {model.rsquared_adj:.4f}")
print(f"  F-stat p-value: {model.f_pvalue:.6f}")

# ─────────────────────────────────────────────────────────────────────────────
#  BONUS — STATISTICAL DIAGNOSTICS
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("BONUS — STATISTICAL DIAGNOSTICS")
print("="*60)

# VIF
vif_data = pd.DataFrame({
    'Feature': feature_cols,
    'VIF':     [variance_inflation_factor(X_const.values, i+1) for i in range(len(feature_cols))]
}).sort_values('VIF', ascending=False)
print("\n[VIF — Multicollinearity Check]")
print(vif_data.head(10).to_string(index=False))
print("  VIF > 10 = severe multicollinearity | VIF 5-10 = moderate")

# Outliers via Z-score
df['z_views'] = np.abs(stats.zscore(df['views']))
n_outliers = (df['z_views'] > 3).sum()
print(f"\n[OUTLIERS] Posts with |Z-score| > 3: {n_outliers} ({n_outliers/len(df)*100:.1f}%)")

# ── Fig 5 : Residual Diagnostics ─────────────────────────────────────────────
fitted    = model.fittedvalues
residuals = model.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Bonus — Regression Diagnostics', fontsize=15, fontweight='bold')

# 1. Residuals vs Fitted
ax = axes[0,0]
ax.scatter(fitted, residuals, alpha=0.35, color=BLUE, s=15, edgecolors='none')
ax.axhline(0, color=CORAL, lw=2, linestyle='--')
ax.set_title('Residuals vs Fitted')
ax.set_xlabel('Fitted values'); ax.set_ylabel('Residuals')
ax.text(0.02, 0.97, 'Want: random scatter around 0\n(homoscedasticity)',
        transform=ax.transAxes, va='top', fontsize=8, color=GRAY)

# 2. Q-Q Plot (normality of residuals)
ax2 = axes[0,1]
stats.probplot(residuals, dist='norm', plot=ax2)
ax2.get_lines()[0].set(color=BLUE, markersize=3, alpha=0.5)
ax2.get_lines()[1].set(color=CORAL, lw=2)
ax2.set_title('Q-Q Plot of Residuals')
ax2.text(0.02, 0.97, 'Want: points on red line\n(normally distributed residuals)',
         transform=ax2.transAxes, va='top', fontsize=8, color=GRAY)

# 3. Scale-Location
ax3 = axes[1,0]
ax3.scatter(fitted, np.sqrt(np.abs(residuals)), alpha=0.35, color=TEAL, s=15, edgecolors='none')
ax3.set_title('Scale-Location (Spread-Location)')
ax3.set_xlabel('Fitted values'); ax3.set_ylabel('√|Residuals|')
ax3.text(0.02, 0.97, 'Want: flat horizontal band',
         transform=ax3.transAxes, va='top', fontsize=8, color=GRAY)

# 4. VIF Bar Chart
ax4 = axes[1,1]
top_vif = vif_data.head(10)
colors_v = [CORAL if v>10 else (AMBER if v>5 else TEAL) for v in top_vif['VIF']]
ax4.barh(top_vif['Feature'], top_vif['VIF'], color=colors_v, edgecolor='white', linewidth=0.5)
ax4.axvline(5,  color=AMBER, lw=1.5, linestyle='--', label='Moderate (5)')
ax4.axvline(10, color=CORAL, lw=1.5, linestyle='--', label='Severe (10)')
ax4.set_title('VIF — Multicollinearity Check')
ax4.set_xlabel('VIF Value')
ax4.legend(fontsize=8)

plt.tight_layout()
plt.savefig('fig5_diagnostics.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 5 saved")

# ── Fig 6 : Outlier Boxplots ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Bonus — Outlier Analysis', fontsize=15, fontweight='bold')

ax = axes[0]
bp = ax.boxplot([df[df['platform']==p]['views'].values/1e6 for p in df['platform'].unique()],
                labels=df['platform'].unique(), patch_artist=True,
                boxprops=dict(facecolor=BLUE, alpha=0.6),
                medianprops=dict(color=CORAL, lw=2),
                flierprops=dict(marker='o', color=CORAL, alpha=0.4, markersize=4))
ax.set_title('Views by Platform (with outliers)')
ax.set_ylabel('Views (Millions)')
ax.tick_params(axis='x', rotation=15)

ax2 = axes[1]
ax2.scatter(df['likes']/1e6, df['views']/1e6,
            c=['red' if z>3 else BLUE for z in df['z_views']],
            alpha=0.4, s=12, edgecolors='none')
ax2.set_title('Views vs Likes (red = Z-score outliers)')
ax2.set_xlabel('Likes (Millions)'); ax2.set_ylabel('Views (Millions)')
from matplotlib.lines import Line2D
ax2.legend(handles=[
    Line2D([0],[0],marker='o',color='w',markerfacecolor=BLUE,markersize=8,label='Normal'),
    Line2D([0],[0],marker='o',color='w',markerfacecolor='red', markersize=8,label='Outlier (|Z|>3)')
], fontsize=9)

plt.tight_layout()
plt.savefig('fig6_outliers.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 6 saved")

print("\n✅ ALL DONE — 6 figures saved")

Dataset loaded: (2000, 18)
post_id                    object
platform                   object
content_type               object
topic                      object
language                   object
region                     object
post_datetime      datetime64[ns]
hashtags                   object
views                       int64
likes                       int64
comments                    int64
shares                      int64
engagement_rate           float64
sentiment_score           float64
is_viral                    int64
hour                        int32
day_of_week                object
log_views                 float64
dtype: object
✔ Fig 1 saved
✔ Fig 2 saved
✔ Fig 3 saved

PHASE 2 — INFERENTIAL ANALYTICS

[T-TEST] Video vs Non-Video views
  H₀: Mean views are equal for video and non-video content
  H₁: Mean views differ between video and non-video content
  Video mean:     4,532,330
  Non-video mean: 4,208,416
  t-statistic: 1.8961  |  p-value: 0.0581
  Result: FAIL to re

In [7]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                             r2_score, mean_absolute_percentage_error)
from sklearn.inspection import permutation_importance
from xgboost import XGBRegressor

# ── Styling ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA', 'axes.facecolor': '#F4F4F0',
    'axes.grid': True, 'grid.color': 'white', 'grid.linewidth': 1.2,
    'font.family': 'DejaVu Sans', 'axes.spines.top': False,
    'axes.spines.right': False,
})
BLUE='#2563EB'; PURPLE='#7C3AED'; CORAL='#E85D2F'
TEAL='#0F766E'; AMBER='#D97706'; GRAY='#6B7280'; GREEN='#16A34A'
COLORS = [BLUE, CORAL, TEAL, PURPLE, AMBER, GREEN]

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 1 — FEATURE ENGINEERING
# ══════════════════════════════════════════════════════════════════════════════
print("="*60)
print("STEP 1 — FEATURE ENGINEERING")
print("="*60)

df = pd.read_csv('raw.csv')
df['post_datetime'] = pd.to_datetime(df['post_datetime'])

# Time features
df['hour']         = df['post_datetime'].dt.hour
df['day_of_week']  = df['post_datetime'].dt.dayofweek      # 0=Mon
df['month']        = df['post_datetime'].dt.month
df['is_weekend']   = df['day_of_week'].isin([5,6]).astype(int)

# Hashtag features
df['hashtag_count'] = df['hashtags'].str.split().str.len()
df['has_viral_tag'] = df['hashtags'].str.contains('#viral|#fyp|#trending',
                                                   case=False, na=False).astype(int)
df['has_ai_tag']    = df['hashtags'].str.contains('#ai', case=False, na=False).astype(int)

# Engagement ratios
df['like_ratio']    = df['likes']   / (df['views'] + 1)
df['comment_ratio'] = df['comments']/ (df['views'] + 1)
df['share_ratio']   = df['shares']  / (df['views'] + 1)

# Target: log-transform views (handles skewness)
df['log_views'] = np.log1p(df['views'])

print(f"Original features:    15")
print(f"Engineered features:  {len(df.columns)-15}")
print(f"Total features:       {len(df.columns)}")
print("\nNew features created:")
new_feats = ['hour','day_of_week','month','is_weekend',
             'hashtag_count','has_viral_tag','has_ai_tag',
             'like_ratio','comment_ratio','share_ratio']
for f in new_feats:
    print(f"  + {f:<20s}  sample: {df[f].iloc[:3].tolist()}")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 2 — PREPROCESSING & TRAIN/TEST SPLIT
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 2 — PREPROCESSING & TRAIN/TEST SPLIT")
print("="*60)

# Encode categoricals
cat_cols = ['platform', 'content_type', 'topic', 'region', 'language']
df_enc   = pd.get_dummies(df, columns=cat_cols, drop_first=True)
bool_cols = df_enc.select_dtypes(include='bool').columns
df_enc[bool_cols] = df_enc[bool_cols].astype(int)

# Feature set — exclude raw target, post_id, datetime, hashtags text, log_views
drop_cols = ['post_id','post_datetime','hashtags','views','log_views',
             'engagement_rate',   # derived from views → data leakage
             'like_ratio','comment_ratio','share_ratio']  # also leakage

FEATURE_COLS = [c for c in df_enc.columns if c not in drop_cols]
TARGET       = 'log_views'

X = df_enc[FEATURE_COLS]
y = df_enc[TARGET]

# 80/20 split, stratified by is_viral
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=df_enc['is_viral']
)

# Scale for linear models
scaler  = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f"Total samples:  {len(df)}")
print(f"Train set:      {len(X_train)} ({len(X_train)/len(df)*100:.0f}%)")
print(f"Test  set:      {len(X_test)}  ({len(X_test)/len(df)*100:.0f}%)")
print(f"Features used:  {len(FEATURE_COLS)}")
print(f"Viral in train: {y_train[X_train['is_viral']==1].count()} | "
      f"Non-viral: {y_train[X_train['is_viral']==0].count()}")
print(f"\nFeature list:\n  " + "\n  ".join(FEATURE_COLS))

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 3 — MODEL TRAINING (4 models)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 3 — MODEL TRAINING")
print("="*60)

models = {
    'Linear Regression':  (LinearRegression(),          X_train_s, X_test_s),
    'Ridge Regression':   (Ridge(alpha=1.0),             X_train_s, X_test_s),
    'Random Forest':      (RandomForestRegressor(
                               n_estimators=200, max_depth=15,
                               min_samples_split=5, random_state=42, n_jobs=-1),
                           X_train, X_test),
    'XGBoost':            (XGBRegressor(
                               n_estimators=300, max_depth=6,
                               learning_rate=0.05, subsample=0.8,
                               colsample_bytree=0.8, random_state=42,
                               verbosity=0),
                           X_train, X_test),
}

results  = {}
kf       = KFold(n_splits=5, shuffle=True, random_state=42)

for name, (model, Xtr, Xte) in models.items():
    print(f"\n  Training {name}...")
    model.fit(Xtr, y_train)
    y_pred = model.predict(Xte)

    # Convert back from log-space for interpretable metrics
    y_test_raw  = np.expm1(y_test)
    y_pred_raw  = np.expm1(y_pred)

    # CV score on log target
    cv_scores = cross_val_score(model, Xtr, y_train,
                                cv=kf, scoring='r2', n_jobs=-1)

    results[name] = {
        'model':   model,
        'y_pred':  y_pred,
        'R2':      r2_score(y_test, y_pred),
        'RMSE':    np.sqrt(mean_squared_error(y_test, y_pred)),
        'MAE':     mean_absolute_error(y_test, y_pred),
        'RMSE_raw':np.sqrt(mean_squared_error(y_test_raw, y_pred_raw)),
        'MAE_raw': mean_absolute_error(y_test_raw, y_pred_raw),
        'CV_mean': cv_scores.mean(),
        'CV_std':  cv_scores.std(),
    }
    r = results[name]
    print(f"    R²={r['R2']:.4f}  RMSE={r['RMSE']:.4f}  "
          f"MAE={r['MAE']:.4f}  CV={r['CV_mean']:.4f}±{r['CV_std']:.4f}")

# ── Summary table ─────────────────────────────────────────────────────────────
print("\n── MODEL COMPARISON TABLE ───────────────────────────────")
print(f"{'Model':<22} {'R²':>7} {'RMSE':>8} {'MAE':>8} {'CV R²':>10} {'CV±':>7}")
print("-"*65)
for name, r in results.items():
    print(f"{name:<22} {r['R2']:>7.4f} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} "
          f"{r['CV_mean']:>10.4f} {r['CV_std']:>7.4f}")

best_name = max(results, key=lambda n: results[n]['R2'])
print(f"\n  ★ Best model: {best_name}  (R² = {results[best_name]['R2']:.4f})")

# ── Fig 7 : Model Comparison ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Step 3 — Model Comparison', fontsize=15, fontweight='bold')
names  = list(results.keys())
r2s    = [results[n]['R2']      for n in names]
rmses  = [results[n]['RMSE']    for n in names]
cvs    = [results[n]['CV_mean'] for n in names]
cvstds = [results[n]['CV_std']  for n in names]

ax = axes[0]
bars = ax.bar(names, r2s, color=COLORS[:len(names)], edgecolor='white', linewidth=0.5)
ax.set_title('R² Score (higher = better)')
ax.set_ylabel('R²'); ax.set_ylim(0, 1.05)
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, r2s):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax2 = axes[1]
bars2 = ax2.bar(names, rmses, color=COLORS[:len(names)], edgecolor='white', linewidth=0.5)
ax2.set_title('RMSE — log(views) (lower = better)')
ax2.set_ylabel('RMSE'); ax2.tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, rmses):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.3f}', ha='center', fontsize=9, fontweight='bold')

ax3 = axes[2]
ax3.bar(names, cvs, color=COLORS[:len(names)], edgecolor='white', linewidth=0.5)
ax3.errorbar(names, cvs, yerr=cvstds, fmt='none', color='black', capsize=5, lw=2)
ax3.set_title('5-Fold Cross-Validation R²')
ax3.set_ylabel('CV R²'); ax3.set_ylim(0, 1.05)
ax3.tick_params(axis='x', rotation=20)
for i,(v,s) in enumerate(zip(cvs, cvstds)):
    ax3.text(i, v+s+0.01, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('fig7_model_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 7 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 4 — BEST MODEL DEEP DIVE (Actual vs Predicted + Residuals)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 4 — BEST MODEL DEEP DIVE")
print("="*60)

best = results[best_name]
y_pred_best = best['y_pred']
residuals   = y_test.values - y_pred_best

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle(f'Step 4 — {best_name}: Actual vs Predicted & Residuals',
             fontsize=15, fontweight='bold')

# 1. Actual vs Predicted (log scale)
ax = axes[0,0]
ax.scatter(y_test, y_pred_best, alpha=0.35, color=BLUE, s=15, edgecolors='none')
lims = [min(y_test.min(), y_pred_best.min())-0.2,
        max(y_test.max(), y_pred_best.max())+0.2]
ax.plot(lims, lims, color=CORAL, lw=2, linestyle='--', label='Perfect prediction')
ax.set_title('Actual vs Predicted (log views)')
ax.set_xlabel('Actual log(views)'); ax.set_ylabel('Predicted log(views)')
ax.legend(fontsize=9)
ax.text(0.05, 0.95, f'R² = {best["R2"]:.4f}', transform=ax.transAxes,
        va='top', fontsize=10, fontweight='bold', color=BLUE)

# 2. Actual vs Predicted (raw views)
ax2 = axes[0,1]
y_raw  = np.expm1(y_test.values)/1e6
yp_raw = np.expm1(y_pred_best)/1e6
ax2.scatter(y_raw, yp_raw, alpha=0.35, color=TEAL, s=15, edgecolors='none')
lims2 = [0, max(y_raw.max(), yp_raw.max())+0.5]
ax2.plot(lims2, lims2, color=CORAL, lw=2, linestyle='--')
ax2.set_title('Actual vs Predicted (raw views in millions)')
ax2.set_xlabel('Actual Views (M)'); ax2.set_ylabel('Predicted Views (M)')
ax2.text(0.05, 0.95, f'RMSE = {best["RMSE_raw"]/1e6:.2f}M\nMAE  = {best["MAE_raw"]/1e6:.2f}M',
         transform=ax2.transAxes, va='top', fontsize=9, family='monospace',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# 3. Residuals distribution
ax3 = axes[1,0]
ax3.hist(residuals, bins=50, color=PURPLE, edgecolor='white', linewidth=0.5, density=True)
x_range = np.linspace(residuals.min(), residuals.max(), 200)
from scipy import stats as sps
ax3.plot(x_range, sps.norm.pdf(x_range, residuals.mean(), residuals.std()),
         color=CORAL, lw=2, label='Normal curve')
ax3.axvline(0, color=AMBER, lw=1.5, linestyle='--')
ax3.set_title('Residuals Distribution')
ax3.set_xlabel('Residual (Actual − Predicted)'); ax3.set_ylabel('Density')
ax3.legend(fontsize=9)
ax3.text(0.97, 0.97,
         f'Mean: {residuals.mean():.4f}\nStd:  {residuals.std():.4f}',
         transform=ax3.transAxes, va='top', ha='right', fontsize=9,
         family='monospace',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

# 4. Residuals vs Fitted (homoscedasticity check)
ax4 = axes[1,1]
ax4.scatter(y_pred_best, residuals, alpha=0.35, color=AMBER, s=15, edgecolors='none')
ax4.axhline(0, color=CORAL, lw=2, linestyle='--')
ax4.set_title('Residuals vs Fitted (homoscedasticity check)')
ax4.set_xlabel('Fitted values'); ax4.set_ylabel('Residuals')
ax4.text(0.02, 0.97,
         'Want: random scatter around 0\n(equal variance = homoscedastic)',
         transform=ax4.transAxes, va='top', fontsize=8, color=GRAY)

plt.tight_layout()
plt.savefig('fig8_actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.close()
print("✔ Fig 8 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 5 — FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 5 — FEATURE IMPORTANCE")
print("="*60)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Step 5 — Feature Importance', fontsize=15, fontweight='bold')

# Random Forest built-in importance
rf_model = results['Random Forest']['model']
rf_imp   = pd.Series(rf_model.feature_importances_, index=FEATURE_COLS)\
             .sort_values(ascending=False).head(15)

ax = axes[0]
colors_imp = [CORAL if i==0 else (BLUE if i<5 else TEAL) for i in range(len(rf_imp))]
ax.barh(rf_imp.index[::-1], rf_imp.values[::-1], color=colors_imp[::-1],
        edgecolor='white', linewidth=0.5)
ax.set_title('Random Forest — Top 15 Feature Importances')
ax.set_xlabel('Importance Score')
for i, (idx, val) in enumerate(zip(rf_imp.index[::-1], rf_imp.values[::-1])):
    ax.text(val+0.001, i, f'{val:.4f}', va='center', fontsize=8)

# XGBoost feature importance
xgb_model = results['XGBoost']['model']
xgb_imp   = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)\
              .sort_values(ascending=False).head(15)

ax2 = axes[1]
colors_xgb = [CORAL if i==0 else (PURPLE if i<5 else AMBER) for i in range(len(xgb_imp))]
ax2.barh(xgb_imp.index[::-1], xgb_imp.values[::-1], color=colors_xgb[::-1],
         edgecolor='white', linewidth=0.5)
ax2.set_title('XGBoost — Top 15 Feature Importances')
ax2.set_xlabel('Importance Score')
for i, (idx, val) in enumerate(zip(xgb_imp.index[::-1], xgb_imp.values[::-1])):
    ax2.text(val+0.001, i, f'{val:.4f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('fig9_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()

print("\nTop 10 features (Random Forest):")
print(rf_imp.head(10).to_string())
print("✔ Fig 9 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 6 — PREDICTION ON NEW DATA (How to use the model)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 6 — PREDICTING ON NEW POSTS (example)")
print("="*60)

# Example: predict views for 3 hypothetical new posts
example_posts = pd.DataFrame({
    'likes':          [500000, 50000, 10000],
    'comments':       [30000,  5000,  500],
    'shares':         [80000,  10000, 1000],
    'sentiment_score':[ 0.8,    0.1,  -0.3],
    'is_viral':       [1,       0,     0],
    'hour':           [18,      12,    9],
    'day_of_week':    [5,       2,     1],
    'month':          [12,      6,     3],
    'is_weekend':     [1,       0,     0],
    'hashtag_count':  [5,       3,     1],
    'has_viral_tag':  [1,       1,     0],
    'has_ai_tag':     [1,       0,     0],
})

# Add all dummy columns with 0s (to match training shape)
for col in FEATURE_COLS:
    if col not in example_posts.columns:
        example_posts[col] = 0

example_posts = example_posts[FEATURE_COLS]

best_model_obj = results[best_name]['model']
Xex = example_posts if best_name in ['Random Forest','XGBoost'] \
      else scaler.transform(example_posts)

log_preds = best_model_obj.predict(Xex)
view_preds = np.expm1(log_preds)

print(f"\nUsing model: {best_name}")
print(f"{'Post':<8} {'Predicted log(views)':>22} {'Predicted Views':>18}")
print("-"*52)
for i,(lp,vp) in enumerate(zip(log_preds, view_preds)):
    print(f"Post {i+1:<3}  {lp:>22.4f}  {vp:>15,.0f}  ({vp/1e6:.2f}M)")

# ══════════════════════════════════════════════════════════════════════════════
#  STEP 7 — LEARNING CURVE (Is more data helpful?)
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("STEP 7 — LEARNING CURVE")
print("="*60)

from sklearn.model_selection import learning_curve

train_sizes, train_scores, val_scores = learning_curve(
    RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    X_train, y_train,
    train_sizes=np.linspace(0.1, 1.0, 10),
    cv=5, scoring='r2', n_jobs=-1
)

fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Step 7 — Learning Curve (Random Forest)', fontsize=15, fontweight='bold')

tr_mean = train_scores.mean(axis=1)
tr_std  = train_scores.std(axis=1)
va_mean = val_scores.mean(axis=1)
va_std  = val_scores.std(axis=1)

ax.plot(train_sizes, tr_mean, 'o-', color=BLUE,   lw=2, label='Training score')
ax.fill_between(train_sizes, tr_mean-tr_std, tr_mean+tr_std, alpha=0.15, color=BLUE)
ax.plot(train_sizes, va_mean, 'o-', color=CORAL,  lw=2, label='Validation score')
ax.fill_between(train_sizes, va_mean-va_std, va_mean+va_std, alpha=0.15, color=CORAL)
ax.set_xlabel('Training set size'); ax.set_ylabel('R²')
ax.legend(fontsize=10); ax.set_ylim(-0.05, 1.05)
gap = tr_mean[-1] - va_mean[-1]
ax.text(0.97, 0.05,
        f'Final gap: {gap:.4f}\n{"→ Slight overfitting" if gap>0.1 else "→ Good fit"}',
        transform=ax.transAxes, ha='right', va='bottom', fontsize=9,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.85))

plt.tight_layout()
plt.savefig('fig10_learning_curve.png', dpi=150, bbox_inches='tight')
plt.close()
print(f"  Training R² (final): {tr_mean[-1]:.4f}")
print(f"  Validation R² (final): {va_mean[-1]:.4f}")
print(f"  Gap: {gap:.4f}")
print("✔ Fig 10 saved")

# ══════════════════════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
print(f"\n{'Model':<22} {'R²':>7} {'CV R²':>8} {'RMSE_raw':>14}")
print("-"*55)
for name, r in results.items():
    marker = " ★" if name == best_name else "  "
    print(f"{name:<22} {r['R2']:>7.4f} {r['CV_mean']:>8.4f} "
          f"{r['RMSE_raw']/1e6:>12.2f}M{marker}")

print(f"\n★ Best model : {best_name}")
print(f"  R²         : {results[best_name]['R2']:.4f}  "
      f"({results[best_name]['R2']*100:.1f}% variance explained)")
print(f"  CV R²      : {results[best_name]['CV_mean']:.4f} ± "
      f"{results[best_name]['CV_std']:.4f}")
print(f"  RMSE (raw) : {results[best_name]['RMSE_raw']/1e6:.2f}M views")
print(f"  MAE  (raw) : {results[best_name]['MAE_raw']/1e6:.2f}M views")
print("\n✅ ALL DONE — Figures 7-10 saved")

STEP 1 — FEATURE ENGINEERING
Original features:    15
Engineered features:  11
Total features:       26

New features created:
  + hour                  sample: [0, 0, 0]
  + day_of_week           sample: [1, 6, 4]
  + month                 sample: [12, 10, 5]
  + is_weekend            sample: [0, 1, 0]
  + hashtag_count         sample: [3, 5, 2]
  + has_viral_tag         sample: [0, 1, 0]
  + has_ai_tag            sample: [0, 1, 1]
  + like_ratio            sample: [0.052631556252568344, 0.04347824374178884, 0.08333325405711883]
  + comment_ratio         sample: [0.00681297898368464, 0.004447175753851245, 0.04489824263658737]
  + share_ratio           sample: [0.000371264234490663, 0.021622122030439655, 0.04198341478171611]

STEP 2 — PREPROCESSING & TRAIN/TEST SPLIT
Total samples:  2000
Train set:      1600 (80%)
Test  set:      400  (20%)
Features used:  31
Viral in train: 1118 | Non-viral: 482

Feature list:
  likes
  comments
  shares
  sentiment_score
  is_viral
  hour
  day_of_we